# Phase 2 — Collinearity diagnostic

Notebook 04 has a jointly-collinear pair of FX regressors (`yoy_real_EUR_TRY`, `yoy_real_GBP_TRY`) and statsmodels flags a high condition number on the joint spec. This is the standard symptom of the two real-FX series moving together, forcing OLS to split the elasticity between them in a numerically unstable way.

This notebook isolates the problem and re-fits three alternative specs:

1. **GBP-only** — drop EUR.
2. **EUR-only** — drop GBP.
3. **Average real-FX index** — replace EUR+GBP with their mean.

Comparing the coefficient on the FX regressor across the four specs (including the joint base from notebook 04) tells us whether the EUR/GBP elasticity is a stable economic signal or an artefact of which collinear regressor happens to be in the model.

Method notes (same as notebook 04): HAC with maxlags = 12 (Δ_12 induces an MA(11) error structure), Trends enter as YoY log change at lag 1, shock controls are 12-month pulse dummies.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from pathlib import Path

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'master_monthly.csv',
                 index_col=0, parse_dates=True)
df.index.name = 'date'

# Defensive pulse-dummy derivation for old CSVs.
idx = df.index
if 'war_pulse' not in df.columns:
    df['war_pulse']           = ((idx >= '2022-02-01') & (idx <= '2023-01-01')).astype(int)
    df['mideast_pulse']       = ((idx >= '2023-10-01') & (idx <= '2024-09-01')).astype(int)
    df['covid_crash_pulse']   = ((idx >= '2020-03-01') & (idx <= '2021-02-01')).astype(int)
    df['covid_rebound_pulse'] = ((idx >= '2021-03-01') & (idx <= '2022-02-01')).astype(int)

def yoy(s): return np.log(s) - np.log(s.shift(12))
def yoy_log_safe(s):
    s = s.replace(0, np.nan)
    return np.log(s) - np.log(s.shift(12))

df['yoy_arrivals']     = yoy(df['arrivals_total'])
df['yoy_real_EUR_TRY'] = yoy(df['real_EUR_TRY'])
df['yoy_real_GBP_TRY'] = yoy(df['real_GBP_TRY'])
df['yoy_real_USD_TRY'] = yoy(df['real_USD_TRY'])
df['yoy_trends_DE'] = yoy_log_safe(df['trends_DE_Türkei_Urlaub'])
df['yoy_trends_GB'] = yoy_log_safe(df['trends_GB_Turkey_holiday'])
df['trends_DE_lag1'] = df['yoy_trends_DE'].shift(1)
df['trends_GB_lag1'] = df['yoy_trends_GB'].shift(1)
df['yoy_real_EUGB_avg'] = (df['yoy_real_EUR_TRY'] + df['yoy_real_GBP_TRY']) / 2

## 1. Pair correlations and VIF

If EUR/GBP move together, their pairwise correlation will be near 1 and their VIFs in any joint regression will be large (rule of thumb: VIF > 10 is severe multicollinearity).

In [2]:
fx_cols = ['yoy_real_EUR_TRY', 'yoy_real_GBP_TRY', 'yoy_real_USD_TRY']
print('Pairwise correlations among YoY real-FX series:')
print(df[fx_cols].corr().round(3).to_string())

Pairwise correlations among YoY real-FX series:
                  yoy_real_EUR_TRY  yoy_real_GBP_TRY  yoy_real_USD_TRY
yoy_real_EUR_TRY             1.000             0.864             0.767
yoy_real_GBP_TRY             0.864             1.000             0.797
yoy_real_USD_TRY             0.767             0.797             1.000


In [3]:
# VIF using the same regressor set as notebook 04's base spec.
regressors_base = [
    'yoy_real_EUR_TRY', 'yoy_real_GBP_TRY',
    'trends_DE_lag1', 'trends_GB_lag1',
    'covid_crash_pulse', 'covid_rebound_pulse', 'war_pulse', 'mideast_pulse',
]
sample = df[['yoy_arrivals'] + regressors_base].dropna()
X = sm.add_constant(sample[regressors_base])
vif = pd.DataFrame({
    'regressor': X.columns,
    'VIF':       [round(variance_inflation_factor(X.values, i), 2)
                  for i in range(X.shape[1])],
})
vif

,regressor,VIF
0,const,2.07
1,yoy_real_EUR_TRY,14.97
2,yoy_real_GBP_TRY,15.80
3,trends_DE_lag1,1.63
4,trends_GB_lag1,3.45
5,covid_crash_pulse,1.83
6,covid_rebound_pulse,1.85
7,war_pulse,2.48
8,mideast_pulse,1.28


## 2. Re-fit under three alternatives

Each spec keeps `trends_DE_lag1`, `trends_GB_lag1`, and the four pulse dummies (`covid_crash_pulse`, `covid_rebound_pulse`, `war_pulse`, `mideast_pulse`) and swaps only the FX regressor(s). Newey–West HAC SEs with **maxlags = 12** throughout (matching notebook 04); same sample window per spec (constrained by FX coverage).

In [4]:
def fit_spec(fx_cols_in_spec, label):
    regs = list(fx_cols_in_spec) + [
        'trends_DE_lag1', 'trends_GB_lag1',
        'covid_crash_pulse', 'covid_rebound_pulse', 'war_pulse', 'mideast_pulse',
    ]
    sub = df[['yoy_arrivals'] + regs].dropna()
    X = sm.add_constant(sub[regs])
    y = sub['yoy_arrivals']
    m = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
    return label, sub, m

specs = {
    'base (EUR+GBP)': ['yoy_real_EUR_TRY', 'yoy_real_GBP_TRY'],
    'GBP-only':       ['yoy_real_GBP_TRY'],
    'EUR-only':       ['yoy_real_EUR_TRY'],
    'avg(EUR,GBP)':   ['yoy_real_EUGB_avg'],
}
results = {label: fit_spec(cols, label) for label, cols in specs.items()}

rows = []
for label, (_, sub, m) in results.items():
    row = {
        'spec':       label,
        'N':          int(m.nobs),
        'R2':         round(m.rsquared, 3),
        'AdjR2':      round(m.rsquared_adj, 3),
        'CondNo':     round(m.condition_number, 0),
        'DW':         round(durbin_watson(m.resid), 3),
    }
    for fx_col in ['yoy_real_EUR_TRY', 'yoy_real_GBP_TRY', 'yoy_real_EUGB_avg']:
        if fx_col in m.params.index:
            row[f'b_{fx_col}'] = round(float(m.params[fx_col]), 2)
            row[f'p_{fx_col}'] = round(float(m.pvalues[fx_col]), 3)
        else:
            row[f'b_{fx_col}'] = np.nan
            row[f'p_{fx_col}'] = np.nan
    rows.append(row)
comparison = pd.DataFrame(rows).set_index('spec')
comparison

,N,R2,AdjR2,CondNo,DW,b_yoy_real_EUR_TRY,p_yoy_real_EUR_TRY,b_yoy_real_GBP_TRY,p_yoy_real_GBP_TRY,b_yoy_real_EUGB_avg,p_yoy_real_EUGB_avg
spec,,,,,,,,,,,
base (EUR+GBP),99,0.688,0.660,45.0,1.207,2.05,0.083,-1.34,0.291,NaN,NaN
GBP-only,99,0.684,0.660,11.0,1.210,NaN,NaN,0.70,0.184,NaN,NaN
EUR-only,99,0.686,0.662,10.0,1.217,0.83,0.094,NaN,NaN,NaN,NaN
"avg(EUR,GBP)",99,0.685,0.661,11.0,1.215,NaN,NaN,NaN,NaN,0.79,0.132


## 3. Full OLS summaries — single-FX and average-FX specs

The summaries below are the three specs that drop the collinear pair: GBP-only, EUR-only, and the EUR/GBP average. Compare each coefficient's sign and 95% interval against the joint base spec to see which side the elasticity "belongs" to.

In [5]:
for label in ['GBP-only', 'EUR-only', 'avg(EUR,GBP)']:
    _, sub, m = results[label]
    print(f'=== {label}  N={int(m.nobs)}  R²={m.rsquared:.3f}  AdjR²={m.rsquared_adj:.3f} ===')
    print(m.summary().tables[1])
    print()

=== GBP-only  N=99  R²=0.684  AdjR²=0.660 ===
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.0785      0.055      1.428      0.153      -0.029       0.186
yoy_real_GBP_TRY        0.6973      0.525      1.328      0.184      -0.332       1.727
trends_DE_lag1         -0.4844      0.258     -1.878      0.060      -0.990       0.021
trends_GB_lag1          0.8334      0.374      2.231      0.026       0.101       1.566
covid_crash_pulse      -1.8750      0.398     -4.716      0.000      -2.654      -1.096
covid_rebound_pulse     1.5987      0.448      3.567      0.000       0.720       2.477
war_pulse              -0.2361      0.306     -0.771      0.441      -0.837       0.364
mideast_pulse          -0.0023      0.065     -0.035      0.972      -0.130       0.125

=== EUR-only  N=99  R²=0.686  AdjR²=0.662 ===
                          c

## 4. Verdict

Cell below picks the spec with the lowest condition number and prints a plain-English read on whether the EUR/GBP collinearity was masking a real elasticity or whether the FX channel is genuinely weak at this aggregation.

In [6]:
alpha = 0.05
best = comparison.sort_values('CondNo').index[0]
best_label, (_, _, best_model) = best, results[best]
_, _, best_model = results[best_label]

fx_in_best = [c for c in ['yoy_real_EUR_TRY', 'yoy_real_GBP_TRY', 'yoy_real_EUGB_avg']
              if c in best_model.params.index]
fx = fx_in_best[0]
b, p = best_model.params[fx], best_model.pvalues[fx]
ci_lo, ci_hi = best_model.conf_int().loc[fx]

print(f'Lowest-condition spec: {best_label}  (condition number {best_model.condition_number:.0f})')
print(f'FX regressor: {fx}')
print(f'  beta = {b:+.3f}   p = {p:.4f}   95% CI = [{ci_lo:+.3f}, {ci_hi:+.3f}]')
print()
if p < alpha and b > 0:
    print('Reading: once the collinear EUR/GBP pair is replaced with a single')
    print(f'regressor, the FX coefficient is {b:+.2f} and significant at 5%.')
    print(f'A 1% real TRY depreciation is associated with a {b:+.2f}% change in YoY arrivals.')
    print('The wrong-sign EUR result in notebook 04 was a collinearity artefact.')
elif p < alpha and b < 0:
    print(f'Single-FX spec also yields a negative significant FX coefficient ({b:+.2f}).')
    print('Collinearity was not the cause — the elasticity itself is wrong-signed in this sample.')
    print('Likely culprits: aggregate arrivals masks country-level heterogeneity, or omitted')
    print('variables (Russian relocation flows, regional safety perception) dominate the macro signal.')
else:
    print(f'Single-FX spec yields beta = {b:+.2f}, p = {p:.3f} — not significant at 5%.')
    print('Collinearity was contributing to instability, but the underlying FX signal is also')
    print('weak. To pin the elasticity down, the natural next steps are: (a) per-source-country')
    print('panel regression instead of aggregate arrivals, (b) richer dynamic spec (AR / DL terms),')
    print('(c) longer / wider FX measure (e.g., REER basket against a broader peer set).')

Lowest-condition spec: EUR-only  (condition number 10)
FX regressor: yoy_real_EUR_TRY
  beta = +0.831   p = 0.0940   95% CI = [-0.142, +1.804]

Single-FX spec yields beta = +0.83, p = 0.094 — not significant at 5%.
Collinearity was contributing to instability, but the underlying FX signal is also
weak. To pin the elasticity down, the natural next steps are: (a) per-source-country
panel regression instead of aggregate arrivals, (b) richer dynamic spec (AR / DL terms),
(c) longer / wider FX measure (e.g., REER basket against a broader peer set).


## 5. Ex-COVID robustness for the preferred single-FX spec

If the lowest-condition spec above is GBP-only or EUR-only, we repeat the notebook 04 robustness check on it: drop the COVID window (2020-03 → 2022-02) and re-fit. In the ex-COVID subsample both COVID pulses are all zero, so they are removed from the regressor list; `war_pulse` and `mideast_pulse` are kept.

In [7]:
if best_label in ('GBP-only', 'EUR-only'):
    fx = fx_in_best[0]
    _, sub_full, m_full = results[best_label]
    mask = (sub_full.index >= '2020-03-01') & (sub_full.index <= '2022-02-01')
    sub_exc = sub_full.loc[~mask].copy()
    regs_exc = [fx, 'trends_DE_lag1', 'trends_GB_lag1', 'war_pulse', 'mideast_pulse']
    X_exc = sm.add_constant(sub_exc[regs_exc])
    y_exc = sub_exc['yoy_arrivals']
    m_exc = sm.OLS(y_exc, X_exc).fit(cov_type='HAC', cov_kwds={'maxlags': 12})

    def _row(name, m):
        ci = m.conf_int().loc[fx]
        return {
            'spec':      name,
            'N':         int(m.nobs),
            f'beta_{fx}': round(float(m.params[fx]), 3),
            f'p_{fx}':    round(float(m.pvalues[fx]), 4),
            'ci_lo':     round(float(ci[0]), 3),
            'ci_hi':     round(float(ci[1]), 3),
            'R2':        round(float(m.rsquared), 3),
        }

    cmp_covid = pd.DataFrame([
        _row(f'{best_label} full', m_full),
        _row(f'{best_label} ex-COVID', m_exc),
    ]).set_index('spec')
    print(cmp_covid)
    print()
    b_f, p_f = m_full.params[fx], m_full.pvalues[fx]
    b_e, p_e = m_exc.params[fx],  m_exc.pvalues[fx]
    if p_e < 0.05 and b_e * b_f > 0:
        print('Verdict: the single-FX elasticity survives the ex-COVID cut at 5%.')
    elif b_e * b_f > 0:
        print('Verdict: same sign, but ex-COVID p-value is above 5%. Full-sample significance is')
        print('at least partly COVID-window dependent.')
    else:
        print('Verdict: sign flips or magnitude collapses ex-COVID. Full-sample headline is')
        print('COVID-window dependent and must be reported as such.')
else:
    print(f'Best spec is {best_label!r}; ex-COVID robustness check skipped.')

                    N  beta_yoy_real_EUR_TRY  p_yoy_real_EUR_TRY  ci_lo  \
spec                                                                      
EUR-only full      99                  0.831              0.0940 -0.142   
EUR-only ex-COVID  75                  0.054              0.6371 -0.170   

                   ci_hi     R2  
spec                             
EUR-only full      1.804  0.686  
EUR-only ex-COVID  0.277  0.683  

Verdict: same sign, but ex-COVID p-value is above 5%. Full-sample significance is
at least partly COVID-window dependent.
